In [1]:
from pathlib import Path
import io
import numpy as np
import pandas as pd
import h5py
import zstandard as zstd

# ---------- HDF5 metadata ----------
def read_meta_h5(meta_path: Path):
    """
    Returns (attrs: dict, df: pandas.DataFrame) for meta file.
    Expects dataset 'events' and attrs: n_channels, trace_samples, trace_dtype.
    """
    meta_path = Path(meta_path)
    with h5py.File(meta_path, "r") as f:
        attrs = {k: (v.decode() if isinstance(v, bytes) else v) for k, v in f.attrs.items()}
        data = f["events"][:]   # structured array
    df = pd.DataFrame({
        "x": data["x"], "y": data["y"], "z": data["z"],
        "energy": data["energy"],
        "type_recoil": [s.decode("utf-8") if isinstance(s, (bytes, bytearray)) else str(s)
                        for s in data["type_recoil"]],
        "no_noise": data["no_noise"],
        "quantize": data["quantize"],
    })
    return attrs, df

# ---------- vectorized unshuffle for a whole batch ----------
def _unshuffle_batch(block: bytes, batch_events: int, n_channels: int, trace_samples: int,
                     dtype=np.float16) -> np.ndarray:
    """
    Inverse of the byte-shuffle used when writing traces.
    Input 'block' holds concatenated shuffled traces for 'batch_events'.
    Returns array with shape (batch_events, n_channels, trace_samples).
    """
    dtype = np.dtype(dtype)
    itemsize = dtype.itemsize
    num_elements = n_channels * trace_samples

    u8 = np.frombuffer(block, dtype=np.uint8)
    expected = batch_events * itemsize * num_elements
    if u8.size != expected:
        raise ValueError(f"Unexpected batch size: got {u8.size} bytes, expected {expected}")
    # [B, itemsize, N] -> [B, N, itemsize] -> flatten last 2 dims, then view
    u8 = u8.reshape(batch_events, itemsize, num_elements).swapaxes(1, 2).reshape(batch_events, num_elements * itemsize)
    arr = u8.view(dtype)  # (B, N)
    return arr.reshape(batch_events, n_channels, trace_samples)

# ---------- single-file batched iterator ----------
def iter_traces_zst_batched_merged(traces_path: Path,
                                   n_events: int,
                                   n_channels: int,
                                   trace_samples: int,
                                   batch_size: int = 1000,
                                   dtype=np.float16,
                                   max_events: int | None = None):
    """
    Iterate over a merged .zst file in *batches*, yielding arrays of shape
    (B, n_channels, trace_samples) where B <= batch_size.
    - Works on a *single* large file written by concatenating event frames.
    - Uses zstd streaming + vectorized unshuffle for speed.
    - If max_events is set, stops after ~max_events (last batch may overrun slightly).
    """
    dtype = np.dtype(dtype)
    per_event_bytes = int(n_channels * trace_samples * dtype.itemsize)
    to_read = n_events if max_events is None else min(max_events, n_events)

    dctx = zstd.ZstdDecompressor()
    with open(traces_path, "rb") as fin, dctx.stream_reader(fin) as reader:
        buf = io.BufferedReader(reader)
        remaining = to_read
        while remaining > 0:
            bsz = int(min(batch_size, remaining))
            need = per_event_bytes * bsz
            got, chunks = 0, []
            while got < need:
                chunk = buf.read(need - got)
                if not chunk:
                    raise EOFError(f"Unexpected end of stream: need {need} bytes, got {got} bytes")
                chunks.append(chunk)
                got += len(chunk)
            block = b"".join(chunks)
            yield _unshuffle_batch(block, bsz, n_channels, trace_samples, dtype=dtype)
            remaining -= bsz

# ---------- convenience wrapper ----------
def open_merged_dataset(base_dir: Path, energy: int):
    """
    Reads meta + returns an iterator factory over the merged traces file.
    Usage:
        attrs, meta_df, get_iter = open_merged_dataset(Path(BASE), 100)
        for batch in get_iter(batch_size=1000, dtype=np.float16, max_events=5000):
            ...
    """
    base = Path(base_dir)
    meta_path = base / f"meta_energy_{energy}.h5"
    traces_path = base / f"traces_energy_{energy}.zst"
    attrs, meta_df = read_meta_h5(meta_path)
    n_events = len(meta_df)
    n_channels = int(attrs["n_channels"])
    trace_samples = int(attrs["trace_samples"])
    trace_dtype = np.dtype(attrs.get("trace_dtype", np.float16))

    def get_iter(batch_size=1000, dtype=trace_dtype, max_events=None):
        return iter_traces_zst_batched_merged(traces_path, n_events, n_channels, trace_samples,
                                              batch_size=batch_size, dtype=dtype, max_events=max_events)
    return attrs, meta_df, get_iter


The history saving thread hit an unexpected error (DatabaseError('database disk image is malformed')).History will not be written to the database.


In [ ]:
from pathlib import Path

BASE = Path("/ceph/dwong/work/training_samples/ER/small")  
E = 1000

attrs, meta_df, get_iter = open_merged_dataset(BASE, E)
print("attrs:", attrs)

count = 0
for batch in get_iter(batch_size=100, dtype=np.float16, max_events=5000):
    # Optional: convert to float32 for training
    batch = batch.astype(np.float32, copy=False)  # shape (B, C, T)
    size = batch.nbytes / 1024**2  # MB
    count += len(batch)
    print(f"batch: {batch.shape}, ~{size:.1f} MB, total: {count}")
print("Done; read ~", count, "events")


attrs: {'compression': 'zstd:15', 'n_channels': np.int64(56), 'trace_dtype': 'float16', 'trace_samples': np.int64(65536)}
batch: (100, 56, 65536), ~1400.0 MB, total: 100
batch: (100, 56, 65536), ~1400.0 MB, total: 200
batch: (100, 56, 65536), ~1400.0 MB, total: 300
batch: (100, 56, 65536), ~1400.0 MB, total: 400
batch: (100, 56, 65536), ~1400.0 MB, total: 500
batch: (100, 56, 65536), ~1400.0 MB, total: 600
batch: (100, 56, 65536), ~1400.0 MB, total: 700
batch: (100, 56, 65536), ~1400.0 MB, total: 800
batch: (100, 56, 65536), ~1400.0 MB, total: 900
batch: (100, 56, 65536), ~1400.0 MB, total: 1000
batch: (100, 56, 65536), ~1400.0 MB, total: 1100
batch: (100, 56, 65536), ~1400.0 MB, total: 1200
batch: (100, 56, 65536), ~1400.0 MB, total: 1300
batch: (100, 56, 65536), ~1400.0 MB, total: 1400
batch: (100, 56, 65536), ~1400.0 MB, total: 1500
batch: (100, 56, 65536), ~1400.0 MB, total: 1600
batch: (100, 56, 65536), ~1400.0 MB, total: 1700
batch: (100, 56, 65536), ~1400.0 MB, total: 1800
batch

In [3]:
from pathlib import Path
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim


try:
    from TCXFormer_v1 import TCXConfig, TCXFormer
    USE_TCX = True
except Exception:
    USE_TCX = False

BASE = Path("/ceph/dwong/work/training_samples/ER")
E = 1000

attrs, meta_df, get_iter = open_merged_dataset(BASE, E)
print("attrs:", attrs)

# -----------------------------
# Setup: device, model, optim
# -----------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.backends.cuda.matmul.allow_tf32 = True
if device.type == "cuda":
    torch.cuda.empty_cache()

# infer shapes
C = int(attrs["n_channels"])
T = int(attrs["trace_samples"])

if USE_TCX:
    # Adjust channel split to your dataset (e.g., 19+37=56)
    cfg = TCXConfig(
        d_model=256, d_ff=1024, n_head=8,
        n_time_layers=2, n_chan_layers=1,
        per_ch_embed=64, stride_photon=128, stride_phonon=128,
        kernel=9, max_seq_len=T,
        n_ch_photon=19, n_ch_phonon=C - 19,   # <-- tweak if different
        classify_er_nr=False,
    )
    model = TCXFormer(cfg).to(device)
else:
    # Fallback tiny baseline if TCX isn't importable
    class TinyBaseline(nn.Module):
        def __init__(self, C, T, d=128):
            super().__init__()
            self.conv = nn.Conv1d(C, d, kernel_size=9, padding=4)
            self.pool = nn.AdaptiveAvgPool1d(256)
            self.norm = nn.LayerNorm(d)
            self.head_xyz = nn.Linear(d, 3)
            self.head_E = nn.Linear(d, 1)
        def forward(self, x):  # x: (B,C,T)
            h = self.conv(x)                   # (B,d,T)
            h = torch.relu(h)
            h = self.pool(h)                   # (B,d,256)
            h = h.mean(-1)                     # (B,d)
            h = self.norm(h)
            return self.head_xyz(h), self.head_E(h), None
    model = TinyBaseline(C, T).to(device)

optimiser = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=0.01)
scaler = torch.cuda.amp.GradScaler(enabled=(device.type == "cuda"))
mse = nn.MSELoss()

# -----------------------------
# Toy training loop
# -----------------------------
BATCH_SIZE = 1000
MAX_EVENTS = 5000   # ~5 batches at 1000 each
MAX_STEPS = None    # or set e.g. 5 to cap steps regardless of events
LOG_EVERY = 1

count = 0
step = 0
model.train()

for batch in get_iter(batch_size=BATCH_SIZE, dtype=np.float16, max_events=MAX_EVENTS):
    # batch: np.ndarray [B, C, T], float16
    B = len(batch)
    # targets from meta_df (aligned with streaming order)
    meta_slice = meta_df.iloc[count: count + B]
    target_xyz = torch.tensor(meta_slice[["x","y","z"]].values, dtype=torch.float32, device=device)
    target_E   = torch.tensor(meta_slice["energy"].values, dtype=torch.float32, device=device).unsqueeze(-1)

    # inputs to torch
    x = torch.from_numpy(batch).to(device)
    x = x.to(torch.float32, copy=False)  # model friendly dtype (keep fp16 if your model supports it end-to-end)

    optimiser.zero_grad(set_to_none=True)
    with torch.cuda.amp.autocast(enabled=(device.type == "cuda")):
        pred_xyz, pred_E, _ = model(x)  # (B,3), (B,1)
        loss_xyz = mse(pred_xyz, target_xyz)
        loss_E = mse(pred_E, target_E)
        loss = loss_xyz + 0.1 * loss_E

    scaler.scale(loss).backward()
    scaler.step(optimiser)
    scaler.update()

    count += B
    step += 1

    if step % LOG_EVERY == 0:
        print(f"step {step:03d} | events {count} | "
              f"loss={loss.item():.4f} (xyz={loss_xyz.item():.4f}, E={loss_E.item():.4f})")

    if (MAX_STEPS is not None) and (step >= MAX_STEPS):
        break

print(f"Done; trained on ~{count} events in {step} steps.")


attrs: {'compression': 'zstd:15', 'n_channels': np.int64(56), 'trace_dtype': 'float16', 'trace_samples': np.int64(150000)}


/tmp/ipykernel_1017748/4068954134.py:63: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=(device.type == "cuda"))


OutOfMemoryError: CUDA out of memory. Tried to allocate 31.29 GiB. GPU 0 has a total capacity of 44.40 GiB of which 385.00 MiB is free. Process 3122575 has 27.88 GiB memory in use. Including non-PyTorch memory, this process has 16.13 GiB memory in use. Of the allocated memory 15.65 GiB is allocated by PyTorch, and 1.73 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [3]:
from pathlib import Path
import pandas as pd

# assumes read_meta_h5(meta_path) is already defined (from your snippet)

def print_h5_info(meta_path: Path, head_rows: int = 5):
    attrs, df = read_meta_h5(meta_path)

    print(f"\n=== Meta file: {meta_path} ===")
    print("Attributes:")
    for k, v in attrs.items():
        print(f"  - {k}: {v}")

    print("\nEvents dataframe:")
    print(f"  - rows (events): {len(df)}")
    print(f"  - columns: {list(df.columns)}")

    # quick sanity stats
    if "energy" in df.columns:
        e_unique = sorted(pd.Series(df["energy"]).unique().tolist())
        print(f"  - unique energies: {e_unique[:10]}{' ...' if len(e_unique) > 10 else ''}")

    print(f"\nHead ({head_rows} rows):")
    print(df.head(head_rows).to_string(index=False))


if __name__ == "__main__":
    # example usage
    meta_path = Path("/ceph/dwong/work/training_samples/ER/small/meta_energy_100.h5")
    print_h5_info(meta_path, head_rows=5)



=== Meta file: /ceph/dwong/work/training_samples/ER/small/meta_energy_100.h5 ===
Attributes:
  - compression: zstd:15
  - n_channels: 56
  - trace_dtype: float16
  - trace_samples: 65536

Events dataframe:
  - rows (events): 25000
  - columns: ['x', 'y', 'z', 'energy', 'type_recoil', 'no_noise', 'quantize']
  - unique energies: [100.0]

Head (5 rows):
         x           y            z  energy type_recoil  no_noise  quantize
-20.949549 -117.661176 -1703.984473   100.0          ER     False      True
-64.020002   40.744912 -1697.787737   100.0          ER     False      True
-23.509832   97.567703 -1702.781686   100.0          ER     False      True
 79.394314   21.287060 -1699.923744   100.0          ER     False      True
 52.806314  -27.502240 -1709.495750   100.0          ER     False      True


In [2]:
from pathlib import Path
import io
import numpy as np
import pandas as pd
import h5py
import zstandard as zstd
import torch
from torch.utils.data import IterableDataset, DataLoader, get_worker_info

# ---------- HDF5 metadata ----------
def read_meta_h5(meta_path: Path):
    meta_path = Path(meta_path)
    with h5py.File(meta_path, "r") as f:
        attrs = {k: (v.decode() if isinstance(v, bytes) else v) for k, v in f.attrs.items()}
        data = f["events"][:]   # structured array
    df = pd.DataFrame({
        "x": data["x"], "y": data["y"], "z": data["z"],
        "energy": data["energy"],
        "type_recoil": [s.decode("utf-8") if isinstance(s, (bytes, bytearray)) else str(s)
                        for s in data["type_recoil"]],
        "no_noise": data["no_noise"],
        "quantize": data["quantize"],
    })
    return attrs, df

# ---------- vectorized unshuffle for a whole batch ----------
def _unshuffle_batch(block: bytes, batch_events: int, n_channels: int, trace_samples: int,
                     dtype=np.float16) -> np.ndarray:
    dtype = np.dtype(dtype)
    itemsize = dtype.itemsize
    num_elements = n_channels * trace_samples

    u8 = np.frombuffer(block, dtype=np.uint8)
    expected = batch_events * itemsize * num_elements
    if u8.size != expected:
        raise ValueError(f"Unexpected batch size: got {u8.size} bytes, expected {expected}")
    u8 = u8.reshape(batch_events, itemsize, num_elements).swapaxes(1, 2).reshape(batch_events, num_elements * itemsize)
    arr = u8.view(dtype)  # (B, N)
    return arr.reshape(batch_events, n_channels, trace_samples)

# ---------- single-file batched iterator (your original) ----------
def iter_traces_zst_batched_merged(traces_path: Path,
                                   n_events: int,
                                   n_channels: int,
                                   trace_samples: int,
                                   batch_size: int = 1000,
                                   dtype=np.float16,
                                   max_events: int | None = None):
    dtype = np.dtype(dtype)
    per_event_bytes = int(n_channels * trace_samples * dtype.itemsize)
    to_read = n_events if max_events is None else min(max_events, n_events)

    dctx = zstd.ZstdDecompressor()
    with open(traces_path, "rb") as fin, dctx.stream_reader(fin) as reader:
        buf = io.BufferedReader(reader)
        remaining = to_read
        while remaining > 0:
            bsz = int(min(batch_size, remaining))
            need = per_event_bytes * bsz
            got, chunks = 0, []
            while got < need:
                chunk = buf.read(need - got)
                if not chunk:
                    raise EOFError(f"Unexpected end of stream: need {need} bytes, got {got} bytes")
                chunks.append(chunk)
                got += len(chunk)
            block = b"".join(chunks)
            yield _unshuffle_batch(block, bsz, n_channels, trace_samples, dtype=dtype)
            remaining -= bsz

# ---------- convenience wrapper (unchanged) ----------
def open_merged_dataset(base_dir: Path, energy: int):
    base = Path(base_dir)
    meta_path = base / f"meta_energy_{energy}.h5"
    traces_path = base / f"traces_energy_{energy}.zst"
    attrs, meta_df = read_meta_h5(meta_path)
    n_events = len(meta_df)
    n_channels = int(attrs["n_channels"])
    trace_samples = int(attrs["trace_samples"])
    trace_dtype = np.dtype(attrs.get("trace_dtype", np.float16))

    def get_iter(batch_size=1000, dtype=trace_dtype, max_events=None):
        return iter_traces_zst_batched_merged(traces_path, n_events, n_channels, trace_samples,
                                              batch_size=batch_size, dtype=dtype, max_events=max_events)
    return attrs, meta_df, get_iter

# ---------- PyTorch IterableDataset for the merged stream ----------
class MergedZstIterableDataset(IterableDataset):
    """
    Streams batches from a single merged .zst file.
    Use num_workers = 0 or 1 (compressed stream is inherently sequential).
    Yields CPU torch.float32 tensors shaped (B, C, T).
    """
    def __init__(self, base_dir: Path, energy: int,
                 batch_size: int = 1000,
                 out_dtype=np.float32,
                 max_events: int | None = None):
        super().__init__()
        self.base_dir = Path(base_dir)
        self.energy = energy
        self.batch_size = batch_size
        self.max_events = max_events
        self.out_dtype = np.dtype(out_dtype)

        attrs, meta_df, get_iter = open_merged_dataset(self.base_dir, self.energy)
        self._attrs = attrs
        self._meta_len = len(meta_df)
        self._get_iter = get_iter
        self.n_channels = int(attrs["n_channels"])
        self.trace_samples = int(attrs["trace_samples"])
        self.trace_dtype = np.dtype(attrs.get("trace_dtype", np.float16))

    def __iter__(self):
        # Ensure only one worker actually reads; others yield nothing.
        wi = get_worker_info()
        if wi is not None and wi.num_workers > 1 and wi.id != 0:
            return iter(())
        for np_batch in self._get_iter(batch_size=self.batch_size,
                                       dtype=self.trace_dtype,
                                       max_events=self.max_events):
            if self.out_dtype != np_batch.dtype:
                np_batch = np_batch.astype(self.out_dtype, copy=False)
            yield torch.from_numpy(np_batch)

# ---------- Example usage ----------
if __name__ == "__main__":
    BASE = Path("/ceph/dwong/work/training_samples/ER/small")
    E = 1000

    ds = MergedZstIterableDataset(BASE, E, batch_size=100, out_dtype=np.float32, max_events=5000)

    loader = DataLoader(
        ds,
        batch_size=None,          # dataset already yields batches
        num_workers=5,            # keep 0 or 1 for single merged stream
        pin_memory=True,
        prefetch_factor=2,        # ignored when num_workers=0 but fine to leave
        persistent_workers=False
    )

    total = 0
    for cpu_batch in loader:
        # async H2D copy
        batch = cpu_batch.to("cuda", non_blocking=True)
        # ... forward/backward ...
        total += batch.shape[0]
        print(f"batch: {tuple(batch.shape)}, ~{batch.element_size()*batch.nelement()/1024**2:.1f} MB, total: {total}")

    print("Done; read ~", total, "events")


batch: (100, 56, 65536), ~1400.0 MB, total: 100
batch: (100, 56, 65536), ~1400.0 MB, total: 200
batch: (100, 56, 65536), ~1400.0 MB, total: 300
batch: (100, 56, 65536), ~1400.0 MB, total: 400
batch: (100, 56, 65536), ~1400.0 MB, total: 500
batch: (100, 56, 65536), ~1400.0 MB, total: 600
batch: (100, 56, 65536), ~1400.0 MB, total: 700
batch: (100, 56, 65536), ~1400.0 MB, total: 800
batch: (100, 56, 65536), ~1400.0 MB, total: 900
batch: (100, 56, 65536), ~1400.0 MB, total: 1000
batch: (100, 56, 65536), ~1400.0 MB, total: 1100
batch: (100, 56, 65536), ~1400.0 MB, total: 1200
batch: (100, 56, 65536), ~1400.0 MB, total: 1300
batch: (100, 56, 65536), ~1400.0 MB, total: 1400
batch: (100, 56, 65536), ~1400.0 MB, total: 1500
batch: (100, 56, 65536), ~1400.0 MB, total: 1600
batch: (100, 56, 65536), ~1400.0 MB, total: 1700
batch: (100, 56, 65536), ~1400.0 MB, total: 1800
batch: (100, 56, 65536), ~1400.0 MB, total: 1900
batch: (100, 56, 65536), ~1400.0 MB, total: 2000
batch: (100, 56, 65536), ~140